# Dormant Kids Incentive - Create CSV(s)

This scripts reads data from the data warehouse and generates the **CSV file(s)** with dormant kids account IDs eligible for the segment offer.

## 1. Read data from data warehouse

In [1]:
TABLE_NAME = "fy26_dormant_kids_incentive"
print(f"TABLE_NAME: {TABLE_NAME}")

TABLE_NAME: fy26_dormant_kids_incentive


In [2]:
from amphibian import get_data_accessor

da = get_data_accessor(engines=["presto"])

def query(sql):
    return da.fetch_sql(sql=sql)

/Users/sergio.oyola/.pyenv/versions/3.9.13/lib/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


In [3]:
from datetime import date

RUN_DATE = None
run_date = RUN_DATE or date.today()

calendar = query(
    f"""--sql
    SELECT
        CAST(MIN(fiscal_date) AS VARCHAR) AS as_of
    FROM curated_historical.cohorting_calendar
    WHERE DATE_TRUNC('month', fiscal_date) = DATE_TRUNC('month', DATE '{run_date}')
"""
)

as_of = calendar.loc[0, "as_of"]
print(f"run date: {run_date}, as_of partition: {as_of}")

run date: 2026-06-19, as_of partition: 2026-06-01


In [4]:
df = query(
    f"""--sql
    SELECT *
    FROM incentive_allocation.{TABLE_NAME}
    WHERE as_of = '{as_of}'
"""
)

In [5]:
# Confirm length of df
df.shape[0]

412434

## 2. Save csv files in batches of 500k

In [ ]:
import os

path = "/Users/sergio.oyola/Desktop/StitchFix/sao-sfix-ds/automation_kids_dormant_offer/output_files"
os.makedirs(path, exist_ok=True)

for i in range(len(df) // 500000 + 1):
    df.iloc[500000 * i : 500000 * (i + 1)] \
        .to_csv(f"{path}/dormant_kids_offer_{as_of}_{i}.csv", index=False)